# 03 — Model Training & Evaluation

This notebook trains **three classifiers** on the preprocessed hand landmark dataset and compares their performance:

| Model | Library |
|---|---|
| Random Forest | scikit-learn |
| XGBoost | xgboost |
| Neural Network | TensorFlow / Keras |

**Pipeline step:** `data.pickle` → Train/Test Split → Model Training → Evaluation → `model.p`

> **Tip:** To quickly retrain only the Random Forest from the terminal:
> ```bash
> python src/train.py
> ```

## 3.1 Imports & Data Loading

In [ ]:
import os
import pickle
import sys
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

sys.path.insert(0, os.path.abspath('..'))
from src.config import LABELS_DICT, MODEL_PATH, PICKLE_PATH, RANDOM_STATE, TEST_SIZE

# ── Load dataset ──────────────────────────────────────────────────────────────
with open(PICKLE_PATH, 'rb') as f:
    data_dict = pickle.load(f)

data   = np.asarray(data_dict['data'])
labels = np.asarray(data_dict['labels'])

print(f'Samples   : {len(data)}')
print(f'Features  : {data.shape[1]}')
print(f'Classes   : {np.unique(labels)}')

# ── Train / test split ────────────────────────────────────────────────────────
x_train, x_test, y_train_raw, y_test_raw = train_test_split(
    data, labels, test_size=TEST_SIZE, shuffle=True,
    stratify=labels, random_state=RANDOM_STATE
)
print(f'\nTrain: {len(x_train)} samples  |  Test: {len(x_test)} samples')

## 3.2 Model 1 — Random Forest

A Random Forest ensemble of decision trees. Robust, interpretable, and fast to train — a strong baseline for tabular landmark data.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf_model.fit(x_train, y_train_raw)
rf_pred = rf_model.predict(x_test)

rf_accuracy  = accuracy_score(y_test_raw, rf_pred)
rf_precision = precision_score(y_test_raw, rf_pred, average='weighted', zero_division=0)
rf_recall    = recall_score(y_test_raw,    rf_pred, average='weighted', zero_division=0)
rf_f1        = f1_score(y_test_raw,        rf_pred, average='weighted', zero_division=0)

print(f'Random Forest — Accuracy : {rf_accuracy  * 100:.2f}%')
print(f'Random Forest — Precision: {rf_precision:.4f}')
print(f'Random Forest — Recall   : {rf_recall:.4f}')
print(f'Random Forest — F1 Score : {rf_f1:.4f}')

## 3.3 Model 2 — XGBoost

Gradient-boosted trees. Often outperforms Random Forest on structured/tabular data through sequential error correction.

In [ ]:
from xgboost import XGBClassifier

# Encode string labels to integers for XGBoost
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train_raw)
y_test_enc  = le.transform(y_test_raw)

xgb_model = XGBClassifier(eval_metric='mlogloss', random_state=RANDOM_STATE)
xgb_model.fit(x_train, y_train_enc)
xgb_pred_enc = xgb_model.predict(x_test)

xgb_accuracy  = accuracy_score(y_test_enc, xgb_pred_enc)
xgb_precision = precision_score(y_test_enc, xgb_pred_enc, average='weighted', zero_division=0)
xgb_recall    = recall_score(y_test_enc,    xgb_pred_enc, average='weighted', zero_division=0)
xgb_f1        = f1_score(y_test_enc,        xgb_pred_enc, average='weighted', zero_division=0)

print(f'XGBoost — Accuracy : {xgb_accuracy  * 100:.2f}%')
print(f'XGBoost — Precision: {xgb_precision:.4f}')
print(f'XGBoost — Recall   : {xgb_recall:.4f}')
print(f'XGBoost — F1 Score : {xgb_f1:.4f}')

## 3.4 Model 3 — Neural Network (Keras)

A fully-connected feedforward network with Dropout regularisation.
Architecture: `128 → 64 → Dropout(0.3) → 32 → softmax`

In [ ]:
import tensorflow as tf
from sklearn.preprocessing import LabelBinarizer
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.models import Sequential

from src.config import NN_BATCH_SIZE, NN_EPOCHS, NN_VAL_SPLIT

lb = LabelBinarizer()
y_train_bin = lb.fit_transform(y_train_raw)
y_test_bin  = lb.transform(y_test_raw)

tf.random.set_seed(RANDOM_STATE)

nn_model = Sequential([
    Dense(128, input_shape=(x_train.shape[1],), activation='relu'),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(y_train_bin.shape[1], activation='softmax'),
])

nn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
nn_model.summary()

history = nn_model.fit(
    x_train, y_train_bin,
    epochs=NN_EPOCHS,
    batch_size=NN_BATCH_SIZE,
    validation_split=NN_VAL_SPLIT,
    verbose=1
)

y_pred_prob = nn_model.predict(x_test)
y_pred_idx  = np.argmax(y_pred_prob, axis=1)
y_test_idx  = np.argmax(y_test_bin,  axis=1)

nn_accuracy  = accuracy_score(y_test_idx, y_pred_idx)
nn_precision = precision_score(y_test_idx, y_pred_idx, average='weighted', zero_division=0)
nn_recall    = recall_score(y_test_idx,    y_pred_idx, average='weighted', zero_division=0)
nn_f1        = f1_score(y_test_idx,        y_pred_idx, average='weighted', zero_division=0)

print(f'\nNeural Network — Accuracy : {nn_accuracy  * 100:.2f}%')
print(f'Neural Network — Precision: {nn_precision:.4f}')
print(f'Neural Network — Recall   : {nn_recall:.4f}')
print(f'Neural Network — F1 Score : {nn_f1:.4f}')

## 3.5 Training Curves (Neural Network)

Visualise how the Neural Network's loss and accuracy evolved over 50 epochs.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Loss
ax1.plot(history.history['loss'],     label='Train Loss',      linewidth=2)
ax1.plot(history.history['val_loss'], label='Val Loss',        linewidth=2, linestyle='--')
ax1.set_title('Loss vs Epoch', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(alpha=0.3)

# Accuracy
ax2.plot(history.history['accuracy'],     label='Train Accuracy', linewidth=2)
ax2.plot(history.history['val_accuracy'], label='Val Accuracy',   linewidth=2, linestyle='--')
ax2.set_title('Accuracy vs Epoch', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle('Neural Network — Training History', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('../assets/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.6 Model Comparison Summary

In [ ]:
print('=' * 58)
print(f'{"Model":<20} {"Accuracy":>10} {"Precision":>10} {"F1":>10}')
print('-' * 58)
print(f'{"Random Forest":<20} {rf_accuracy*100:>9.2f}% {rf_precision:>10.4f} {rf_f1:>10.4f}')
print(f'{"XGBoost":<20} {xgb_accuracy*100:>9.2f}% {xgb_precision:>10.4f} {xgb_f1:>10.4f}')
print(f'{"Neural Network":<20} {nn_accuracy*100:>9.2f}% {nn_precision:>10.4f} {nn_f1:>10.4f}')
print('=' * 58)

## 3.7 Confusion Matrix (Best Model)

Random Forest is saved as the production model. Let's inspect its confusion matrix on the test set.

In [ ]:
class_names = [LABELS_DICT.get(int(k), k) for k in sorted(np.unique(labels))]
cm = confusion_matrix(y_test_raw, rf_pred, labels=sorted(np.unique(labels)))

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.5, ax=ax)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Confusion Matrix — Random Forest', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../assets/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nClassification Report (Random Forest):')
print(classification_report(y_test_raw, rf_pred,
      target_names=class_names, zero_division=0))

## 3.8 Save Best Model

The **Random Forest** is selected as the production model and saved to `models/model.p`.

In [ ]:
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

with open(MODEL_PATH, 'wb') as f:
    pickle.dump({'model': rf_model}, f)

print(f'Best model (Random Forest) saved → {MODEL_PATH}')